# 01 - Create and Expand a Domain Red-Team Dataset

This notebook builds the benchmark dataset from scratch.

It is designed as a reusable utility notebook. The current project implementation is finance-specific, but the notebook keeps the domain settings in one place so you can adapt the workflow for another domain later by swapping taxonomy, seed prompts, and mappings.

Pipeline:

```text
Domain config -> taxonomy -> seed prompts -> local records -> DeepTeam expansion -> Garak patterns -> normalize -> deduplicate -> validate -> export JSONL + Promptfoo YAML
```

Default output:

```text
data/exports/finance_redteam_attacks.jsonl
data/exports/promptfoo_tests.yaml
```

## 1. Setup

Run this notebook from the project root, or let the cell below locate the project root automatically.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "finance_redteam").exists():
    candidates = [p for p in Path.cwd().parents if (p / "src" / "finance_redteam").exists()]
    if not candidates:
        raise RuntimeError("Could not find project root. Open this notebook from finance-llm-redteam-benchmark.")
    PROJECT_ROOT = candidates[0]

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

## 2. Domain Configuration

The notebook reads the same YAML config as the CLI and scripts. Edit `configs/finance_benchmark.yaml` for reproducible production runs.

For a future domain, use a new config file plus domain-specific taxonomy, seed prompt builder, mappings, and exporter naming conventions.

In [ ]:
from finance_redteam.config import DEFAULT_CONFIG_PATH, load_benchmark_config

config = load_benchmark_config(DEFAULT_CONFIG_PATH)
config

## 3. Generate Taxonomy and Seed Prompts

This uses the existing project modules. It does not rewrite package code.

In [ ]:
from finance_redteam.taxonomy import (
    DEFAULT_FRAMEWORK_MAPPINGS_PATH,
    DEFAULT_TAXONOMY_PATH,
    load_taxonomy,
    write_framework_mappings,
    write_taxonomy,
)
from finance_redteam.seed_prompts import DEFAULT_SEEDS_PATH, build_seed_prompts, write_seed_prompts

write_taxonomy(DEFAULT_TAXONOMY_PATH)
write_framework_mappings(DEFAULT_FRAMEWORK_MAPPINGS_PATH)
write_seed_prompts(DEFAULT_SEEDS_PATH, per_category=config.min_per_category)

categories = load_taxonomy()
seeds = build_seed_prompts(categories, per_category=config.min_per_category)

print(f"Taxonomy categories: {len(categories)}")
print(f"Seed prompts: {len(seeds)}")
print(f"Taxonomy path: {DEFAULT_TAXONOMY_PATH}")
print(f"Seeds path: {DEFAULT_SEEDS_PATH}")

## 4. Build Base Records

The base dataset comes from the static finance taxonomy and seed prompts.

In [ ]:
from finance_redteam.local_generator import generate_local_variants, seeds_to_records

records = seeds_to_records(seeds, categories, source="seed")

if len(records) < len(categories) * config.min_per_category:
    records.extend(generate_local_variants(categories, per_category=1, start_index=900))

print(f"Base records: {len(records)}")
records[0].model_dump()

## 5. Optional DeepTeam Expansion

DeepTeam adds adversarial variations when installed. If it is missing or cannot run, the adapter returns an empty list and the pipeline continues.

Use the config values above to choose vulnerability families, attack styles, category filters, and difficulty bounds. Keep prompts framed as defensive evaluation inputs; the expected behavior remains refusal, safe redirection, or compliance-safe guidance.

In [ ]:
from finance_redteam.deepteam_adapter import (
    DeepTeamExpansionConfig,
    generate_deepteam_variants,
    is_deepteam_available,
    supported_deepteam_attack_types,
    supported_deepteam_vulnerabilities,
)
from finance_redteam.exporters import export_jsonl

deepteam_config = DeepTeamExpansionConfig(
    vulnerabilities=list(config.deepteam.vulnerabilities),
    attack_types=list(config.deepteam.attack_types),
    risk_categories=list(config.deepteam.risk_categories),
    min_difficulty=config.deepteam.min_difficulty,
    max_difficulty=config.deepteam.max_difficulty,
    max_seed_records=config.deepteam.max_seed_records,
    variants_per_seed=config.deepteam.variants_per_seed,
    include_original_attack_type=config.deepteam.include_original_attack_type,
)

print("Supported DeepTeam vulnerability selectors:")
print(supported_deepteam_vulnerabilities())
print("Supported DeepTeam attack type selectors:")
print(supported_deepteam_attack_types())

deepteam_variants = []
if config.use_deepteam:
    print(f"DeepTeam installed: {is_deepteam_available()}")
    print(f"Selected DeepTeam config: {deepteam_config}")
    deepteam_variants = generate_deepteam_variants(
        records,
        auto_install=False,
        expansion_config=deepteam_config,
    )

export_jsonl(deepteam_variants, Path("data/generated/deepteam_variants.jsonl"))
records.extend(deepteam_variants)

print(f"DeepTeam variants added: {len(deepteam_variants)}")
print(f"Records after DeepTeam: {len(records)}")

## 6. Optional Garak Coverage Expansion

Garak adds scanner-style patterns for coverage. The adapter keeps Garak isolated from the main build.

In [ ]:
from finance_redteam.garak_adapter import is_garak_available, run_garak_scan

garak_records = []
if config.use_garak or config.garak.enabled:
    print(f"Garak installed: {is_garak_available()}")
    garak_records = run_garak_scan(
        target_model=config.garak.target_model,
        auto_install=config.garak.auto_install,
        report_dir=config.garak.report_dir,
        max_findings=config.garak.max_findings,
    )

export_jsonl(garak_records, Path("data/generated/garak_patterns.jsonl"))
records.extend(garak_records)

print(f"Garak records added: {len(garak_records)}")
print(f"Total raw records: {len(records)}")

## 7. Normalize, Deduplicate, and Validate

This is the quality gate before export.

In [ ]:
from finance_redteam.deduplicator import deduplicate_records
from finance_redteam.normalizer import normalize_record
from finance_redteam.provenance import create_generation_run, stamp_records, write_run_metadata
from finance_redteam.validator import validate_records

generation_run = create_generation_run(config)
normalized_records = [normalize_record(record) for record in records]
final_records = stamp_records(deduplicate_records(normalized_records), generation_run)
validation = validate_records(final_records)

print(f"Records before dedupe: {len(records)}")
print(f"Records after dedupe: {len(final_records)}")
print(f"Validation passed: {validation.valid}")

if validation.review_warnings:
    print("Review warnings:")
    for warning in validation.review_warnings[:20]:
        print("-", warning)

if not validation.valid:
    for error in validation.errors[:50]:
        print("ERROR:", error)
    raise RuntimeError("Dataset validation failed")

## 8. Export JSONL and Promptfoo YAML

The exported JSONL is the benchmark dataset. The Promptfoo YAML is used by the evaluation notebook.

In [ ]:
from finance_redteam.promptfoo_exporter import export_promptfoo

export_jsonl(final_records, config.output.jsonl)
export_jsonl(final_records, config.output.normalized_jsonl)
export_promptfoo(final_records, config.output.promptfoo_yaml)
write_run_metadata(config, generation_run, config.output.run_metadata_json, len(final_records))

print(f"Exported {len(final_records)} records to {config.output.jsonl}")
print(f"Exported Promptfoo config to {config.output.promptfoo_yaml}")
print(f"Wrote run metadata to {config.output.run_metadata_json}")

## 9. Preview Exported Dataset

This shows a few records without running any model calls.

In [ ]:
with config.output.jsonl.open("r", encoding="utf-8") as handle:
    preview = [json.loads(next(handle)) for _ in range(3)]

for item in preview:
    print(json.dumps({
        "attack_id": item["attack_id"],
        "risk_category": item["risk_category"],
        "attack_type": item["attack_type"],
        "difficulty": item["difficulty"],
        "source": item["source"],
        "prompt": item["prompt"][:180] + "...",
    }, indent=2))

## 10. Next Step

Run `02_evaluate_attacks_gemini.ipynb` to evaluate the generated attacks against Gemini using Promptfoo.